In [1]:
import pandas as pd
df = pd.read_csv("../data/soc-redditHyperlinks-body.tsv", sep='\t')
df

,SOURCE_SUBREDDIT,TARGET_SUBREDDIT,POST_ID,TIMESTAMP,LINK_SENTIMENT,PROPERTIES
0,leagueoflegends,teamredditteams,1u4nrps,2013-12-31 16:39:58,1,"345.0,298.0,0.75652173913,0.0173913043478,0.08..."
1,theredlion,soccer,1u4qkd,2013-12-31 18:18:37,-1,"101.0,98.0,0.742574257426,0.019801980198,0.049..."
2,inlandempire,bikela,1u4qlzs,2014-01-01 14:54:35,1,"85.0,85.0,0.752941176471,0.0235294117647,0.082..."
3,nfl,cfb,1u4sjvs,2013-12-31 17:37:55,1,"1124.0,949.0,0.772241992883,0.0017793594306,0...."
4,playmygame,gamedev,1u4w5ss,2014-01-01 02:51:13,1,"715.0,622.0,0.777622377622,0.00699300699301,0...."
...,...,...,...,...,...,...
286556,negareddit,debatefascism,68im20s,2017-04-30 16:31:26,1,"441.0,405.0,0.775510204082,0.0294784580499,0.0..."
286557,mildlynomil,justnomil,68imlas,2017-04-30 04:19:03,1,"2226.0,1855.0,0.786163522013,0.00224618149146,..."
286558,mmorpg,blackdesertonline,68ip5os,2017-04-30 16:54:08,1,"1100.0,909.0,0.778181818182,0.00181818181818,0..."
286559,electricskateboards,askreddit,68ipb2s,2017-04-30 16:41:53,1,"1876.0,1567.0,0.78144989339,0.00692963752665,0..."


In [2]:
df = df.drop(['POST_ID','PROPERTIES'], axis=1)
# Save the dataframe to a new CSV file
df.to_csv("../data/redditHyperlinks-column-filtered.csv", index=False)

# Display the updated DataFrame
print(len(df))
df.head()

286561


,SOURCE_SUBREDDIT,TARGET_SUBREDDIT,TIMESTAMP,LINK_SENTIMENT
0,leagueoflegends,teamredditteams,2013-12-31 16:39:58,1
1,theredlion,soccer,2013-12-31 18:18:37,-1
2,inlandempire,bikela,2014-01-01 14:54:35,1
3,nfl,cfb,2013-12-31 17:37:55,1
4,playmygame,gamedev,2014-01-01 02:51:13,1


# Create the graph
We create separate dictionaries for negative and positive edges. In the graph, there should not be duplicate edges. Therefore, we count how many negative and positive mentions there are from one subreddit to another and use this as the edge weight.

In [ ]:
import networkx as nx
import json
from collections import defaultdict
df = pd.read_csv("../data/redditHyperlinks-column-filtered.csv")

# create separate dictionaries for negative and positive edges
positive = defaultdict(int)
negative = defaultdict(int)
for index, row in df.iterrows():
    # count number of mentions between subreddits
    if row['LINK_SENTIMENT'] == 1:
        positive[(row['SOURCE_SUBREDDIT'],row['TARGET_SUBREDDIT'])] += 1
    else:
        negative[(row['SOURCE_SUBREDDIT'],row['TARGET_SUBREDDIT'])] += 1

# create weighted list of edges based on number of mentions
positive_weighted_list=[(sub1,sub2,weight) for (sub1,sub2),weight in positive.items()]
negative_weighted_list=[(sub1,sub2,weight) for (sub1,sub2),weight in negative.items()]  

# create the graphs
G_positive=nx.DiGraph()
G_positive.add_weighted_edges_from(positive_weighted_list)
G_negative=nx.DiGraph()
G_negative.add_weighted_edges_from(negative_weighted_list)

# save the graphs
Json_pos=nx.node_link_data(G_positive)
with open("../data/positive_graph.json",'w') as file:
    json.dump(Json_pos,file,indent=2) 

Json_neg=nx.node_link_data(G_negative)
with open("../data/negative_graph.json",'w') as file:
    json.dump(Json_neg,file,indent=2) 

Now we would like to find communities in both graphs. FORKLAR SENERE

In [10]:
import community as community_louvain
import netwulf as nw

def compute_communities(G,title):
    'Computes the communities and returns a data partition for netwulf visualization'

    partition=community_louvain.best_partition(G)
    for node, data in G.nodes(data=True):
        data['group'] = partition[node]

    # save community partition
    json_data = nx.node_link_data(G)
    with open(f'../data/{title}_communities.json','w') as file:
        json.dump(json_data,file,indent=2)
    
    # convert graph to correct format for plotting
    graph_data_partition=nx.json_graph.node_link_data(G)
    graph_data_partition['links'] = graph_data_partition.pop('edges')
    return graph_data_partition

ModuleNotFoundError: No module named 'distutils'